In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"
scenario = "intervention"

In [3]:
# Parameters
location = "nigeria"
vehicle = "bouillon"
scenario = "intervention"


In [4]:
def aggregate_by_scenario(df):
    return df.groupby(["scenario", "input_draw", "wealth_quintile","sub_entity"]).value.sum().groupby(["scenario", "wealth_quintile","sub_entity"]).mean()

In [5]:
preg_anemia_prev = pd.read_parquet(f"0100_rescale_results/pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet")
preg_anemia_prev

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,lowest,intervention,4,0,49.288956
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,second,intervention,4,0,94.004709
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,middle,intervention,4,0,151.423802
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,fourth,intervention,4,0,153.964469
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,highest,intervention,4,0,119.919521
...,...,...,...,...,...,...,...,...,...,...,...
35995,person_time,impairment,anemia,severe,95_plus,severe,lowest,intervention,3,0,0.000000
35996,person_time,impairment,anemia,severe,95_plus,severe,second,intervention,3,0,0.000000
35997,person_time,impairment,anemia,severe,95_plus,severe,middle,intervention,3,0,0.000000
35998,person_time,impairment,anemia,severe,95_plus,severe,fourth,intervention,3,0,0.000000


In [6]:
preg_anemia_prev.sub_entity.value_counts()
# TODO: Double-check that sub_entity is the best variable to use for anemia status? (Or is anemia_status_by_birth better?) 

mild          9000
moderate      9000
not_anemic    9000
severe        9000
Name: sub_entity, dtype: int64

In [7]:
preg_anemia_prev.anemia_status_at_birth.value_counts()

invalid       7200
mild          7200
moderate      7200
not_anemic    7200
severe        7200
Name: anemia_status_at_birth, dtype: int64

In [8]:
preg_anemia_prev.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
Name: random_seed, dtype: int64

In [9]:
preg_anemia_prev_by_scenario = aggregate_by_scenario(preg_anemia_prev)
preg_anemia_prev_by_scenario

scenario      wealth_quintile  sub_entity
baseline      fourth           mild          159545.808398
                               moderate      166116.991625
                               not_anemic    363957.269673
                               severe         10051.389994
              highest          mild          145863.295969
                               moderate      125580.636699
                               not_anemic    339750.294902
                               severe          5366.398551
              lowest           mild          269926.137718
                               moderate      302742.927566
                               not_anemic    406293.433676
                               severe         15264.840342
              middle           mild          222557.927048
                               moderate      238996.047786
                               not_anemic    384059.541527
                               severe         12892.872874
              

In [10]:
preg_total_person_time = preg_anemia_prev_by_scenario.groupby(level=['scenario', 'wealth_quintile']).sum()
preg_total_person_time

scenario      wealth_quintile
baseline      fourth             6.996715e+05
              highest            6.165606e+05
              lowest             9.942273e+05
              middle             8.585064e+05
              second             1.027322e+06
intervention  fourth             6.996786e+05
              highest            6.165606e+05
              lowest             9.942416e+05
              middle             8.585099e+05
              second             1.027322e+06
Name: value, dtype: float64

In [11]:
preg_anemia_prev_by_scenario_and_state = preg_anemia_prev_by_scenario[preg_anemia_prev_by_scenario.index.get_level_values('sub_entity') != 'not_anemic']
preg_anemia_prev_by_scenario_and_state = preg_anemia_prev_by_scenario_and_state / preg_total_person_time
preg_anemia_prev_by_scenario_and_state
# Sum the person-times of each anemia state together (total person-time), then divide the person-time within each state (sub_entity) by the total person-time across all states
# to get prevalence rate of each anemia state

scenario      wealth_quintile  sub_entity
baseline      fourth           mild          0.228030
                               moderate      0.237421
                               severe        0.014366
              highest          mild          0.236576
                               moderate      0.203679
                               severe        0.008704
              lowest           mild          0.271493
                               moderate      0.304501
                               severe        0.015353
              middle           mild          0.259239
                               moderate      0.278386
                               severe        0.015018
              second           mild          0.259140
                               moderate      0.330042
                               severe        0.020538
intervention  fourth           mild          0.223555
                               moderate      0.227659
                               severe   

In [12]:
# Now that I've calculated the prevalence of each anemia state seaparately, I can sum them together and stratify by just scenario and wealth quintile. 
preg_anemia_prev_by_scenario = preg_anemia_prev_by_scenario_and_state.groupby(level=['scenario', 'wealth_quintile']).sum() 
preg_anemia_prev_by_scenario

scenario      wealth_quintile
baseline      fourth             0.479817
              highest            0.448959
              lowest             0.591348
              middle             0.552642
              second             0.609720
intervention  fourth             0.464917
              highest            0.434374
              lowest             0.565888
              middle             0.533727
              second             0.586848
Name: value, dtype: float64

In [13]:
preg_anemia_prev_averted = preg_anemia_prev_by_scenario.loc["baseline"] - preg_anemia_prev_by_scenario.loc[scenario]
preg_anemia_prev_averted 
# This should be in RATE-space. 

wealth_quintile
fourth     0.014900
highest    0.014585
lowest     0.025460
middle     0.018915
second     0.022872
Name: value, dtype: float64

In [14]:
preg_pct_anemia_prev_averted = (preg_anemia_prev_averted / preg_anemia_prev_by_scenario.loc["baseline"]) * 100 
preg_pct_anemia_prev_averted

wealth_quintile
fourth     3.105358
highest    3.248592
lowest     4.305347
middle     3.422680
second     3.751259
Name: value, dtype: float64

In [15]:
df_preg_anemia = pd.DataFrame(preg_anemia_prev_averted).rename(columns={'value':'anemia_prev_averted'})
df_preg_anemia

,anemia_prev_averted
wealth_quintile,
fourth,0.014900
highest,0.014585
lowest,0.025460
middle,0.018915
second,0.022872


In [16]:
df_preg_anemia['pct_anemia_prev_averted'] = preg_pct_anemia_prev_averted
df_preg_anemia

,anemia_prev_averted,pct_anemia_prev_averted
wealth_quintile,,
fourth,0.014900,3.105358
highest,0.014585,3.248592
lowest,0.025460,4.305347
middle,0.018915,3.422680
second,0.022872,3.751259


In [17]:
df_preg_anemia = df_preg_anemia.assign(pregnant="pregnant").set_index("pregnant", append=True)

In [18]:
non_preg_anemia_prev = pd.read_parquet(f"../0400_non_pregnant_anemia_model/{vehicle}/{location}/{scenario}/anemia_prevalence.parquet")
non_preg_anemia_prev = non_preg_anemia_prev.set_index([c for c in non_preg_anemia_prev.columns if c != "value"]).assign(pregnant="not_pregnant").set_index("pregnant", append=True).value
non_preg_anemia_prev

age_start  age_end     sex     wealth_quintile  scenario      pregnant    
0.0        0.019178    Female  fourth           baseline      not_pregnant    0.828338
                               highest          baseline      not_pregnant    0.748522
                               lowest           baseline      not_pregnant    0.900708
                               middle           baseline      not_pregnant    0.835819
                               second           baseline      not_pregnant    0.876689
                                                                                ...   
95.0       125.000000  Male    fourth           intervention  not_pregnant    0.888494
                               highest          intervention  not_pregnant    0.878136
                               lowest           intervention  not_pregnant    0.904701
                               middle           intervention  not_pregnant    0.887005
                               second           interve

In [19]:
pop = pd.read_csv(f'../0100_data_prep/results/population/stratified/{location}.csv')
pop = pop.set_index([c for c in pop.columns if c != "value"]).value
pop

sex     age_start  age_end     pregnant      wealth_quintile
Female  0.0        0.019178    not_pregnant  lowest             16960.496219
                                             second             16890.884668
                                             middle             15946.960379
                                             fourth             14017.956071
                                             highest            13029.810174
                                                                    ...     
Male    95.0       125.000000  not_pregnant  lowest              1846.968043
                                             second              1616.329970
                                             middle              1661.447043
                                             fourth              1779.062429
                                             highest             1896.973078
Name: value, Length: 285, dtype: float64

In [20]:
non_preg_anemia_prev_by_scenario = (non_preg_anemia_prev * pop).groupby(["scenario", "wealth_quintile"]).sum() / pop.groupby(["wealth_quintile"]).sum()
non_preg_anemia_prev_by_scenario

scenario      wealth_quintile
baseline      fourth             0.423153
              highest            0.342180
              lowest             0.535335
              middle             0.439019
              second             0.479142
intervention  fourth             0.415901
              highest            0.337156
              lowest             0.520771
              middle             0.430458
              second             0.467549
Name: value, dtype: float64

In [21]:
non_preg_anemia_prev_averted = non_preg_anemia_prev_by_scenario.loc["baseline"] - non_preg_anemia_prev_by_scenario.loc["intervention"]
non_preg_anemia_prev_averted

wealth_quintile
fourth     0.007252
highest    0.005024
lowest     0.014565
middle     0.008560
second     0.011592
Name: value, dtype: float64

In [22]:
non_preg_pct_anemia_prev_averted = (non_preg_anemia_prev_averted / non_preg_anemia_prev_by_scenario.loc["baseline"]) * 100 
non_preg_pct_anemia_prev_averted

wealth_quintile
fourth     1.713905
highest    1.468210
lowest     2.720658
middle     1.949859
second     2.419385
Name: value, dtype: float64

In [23]:
df_non_preg_anemia = pd.DataFrame(non_preg_anemia_prev_averted).rename(columns={'value':'anemia_prev_averted'})
df_non_preg_anemia

,anemia_prev_averted
wealth_quintile,
fourth,0.007252
highest,0.005024
lowest,0.014565
middle,0.008560
second,0.011592


In [24]:
df_non_preg_anemia['pct_anemia_prev_averted'] = non_preg_pct_anemia_prev_averted
df_non_preg_anemia

,anemia_prev_averted,pct_anemia_prev_averted
wealth_quintile,,
fourth,0.007252,1.713905
highest,0.005024,1.468210
lowest,0.014565,2.720658
middle,0.008560,1.949859
second,0.011592,2.419385


In [25]:
df_non_preg_anemia = df_non_preg_anemia.assign(pregnant="not_pregnant").set_index("pregnant", append=True)

In [26]:
df_anemia = pd.concat([df_preg_anemia, df_non_preg_anemia]).sort_index()
df_anemia

anemia_prev_averted  pct_anemia_prev_averted
wealth_quintile pregnant                                                  
fourth          not_pregnant             0.007252                 1.713905
                pregnant                 0.014900                 3.105358
highest         not_pregnant             0.005024                 1.468210
                pregnant                 0.014585                 3.248592
lowest          not_pregnant             0.014565                 2.720658
                pregnant                 0.025460                 4.305347
middle          not_pregnant             0.008560                 1.949859
                pregnant                 0.018915                 3.422680
second          not_pregnant             0.011592                 2.419385
                pregnant                 0.022872                 3.751259